In [0]:
import os
import sys

src_path = os.path.abspath("../src")

if src_path not in sys.path:
    sys.path.insert(0, src_path)

from config import BRONZE_TABLE, LANDING_DIR

from bronze import (
    check_idempotency,
    create_bronze_table,
    get_landing_files,
    load_bronze,
    reconcile,
    rows_per_source_file,
)


files = get_landing_files()

if not files:
    raise ValueError("Run 02_seed_landing first.")


print(f"Landing directory: {LANDING_DIR}")
print(f"Bronze table: {BRONZE_TABLE}")
print(f"Landing files: {len(files)}")

In [0]:
create_bronze_table(spark)

print("Bronze table is ready:", BRONZE_TABLE)

In [0]:
load_bronze(spark)

print("Bronze ingestion completed.")

In [0]:
bronze_df = spark.table(BRONZE_TABLE)

print("Bronze schema:")
bronze_df.printSchema()

print("Sample records:")
display(bronze_df.limit(10))

In [0]:
print("Rows per source file:")
display(rows_per_source_file(spark))

In [0]:
reconciliation_passed = reconcile(spark)

if not reconciliation_passed:
    raise ValueError("Bronze reconciliation failed.")


idempotency_passed = check_idempotency(spark)

if not idempotency_passed:
    raise ValueError("Bronze idempotency check failed.")


print("Bronze pipeline completed successfully.")